## Huggingface Dataset to AML Tutorial

The following notebook shows how to construct an AML pipeline that pulls a Huggingface dataset into an AML data asset 

### (Optional) Configure the environment 

In [ ]:
%set_env WORKSPACE_NAME=<WORKSPACE NAME>
%set_env RESOURCE_GROUP_NAME=<RESOURCE GROUP>
%set_env SUBSCRIPTION_ID=<SUBSCRIPTION ID>

!az configure --defaults workspace=%WORKSPACE_NAME% group=%RESOURCE_GROUP_NAME%

### Create a single-node compute cluster

In [12]:
!az ml compute create -n cpu-cluster --type amlcompute \
    --min-instances 0 \
    --max-instances 1 \
    --size STANDARD_DS3_V2 \
    --idle-time-before-scale-down 1800 \
    --tier Dedicated

{
  "enable_node_public_ip": true,
  "id": "/subscriptions/781b03e7-6eb7-4506-bab8-cf3a0d89b1d4/resourceGroups/antonslutsky-rg/providers/Microsoft.MachineLearningServices/workspaces/gpu-workspace-uksouth/computes/cpu-cluster",
  "idle_time_before_scale_down": 1800,
  "location": "uksouth",
  "max_instances": 1,
  "min_instances": 0,
  "name": "cpu-cluster",
  "network_settings": {},
  "provisioning_state": "Succeeded",
  "resourceGroup": "antonslutsky-rg",
  "size": "STANDARD_DS3_V2",
  "ssh_public_access_enabled": true,
  "tier": "dedicated",
  "type": "amlcompute"
}


### Generate a job YAML 

molssiai-hub/pubchemqc-b3lyp dataset is a large collection of JSON files

In [14]:
%%writefile pull_dataset_from_huggingface.yaml
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
code: src
command: >-
  ./pull_dataset_from_huggingface.sh ${{inputs.huggingface_dataset_name}} ${{outputs.output_dir_path}}
inputs:
  huggingface_dataset_name: molssiai-hub/pubchemqc-b3lyp
outputs:
  output_dir_path:
    mode: rw_mount
    type: uri_folder
environment: 
  image: mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04:latest
compute: azureml:cpu-cluster
services:
  my_vs_code:
    type: vs_code
    nodes: all # For distributed jobs, use the `nodes` property to pick which node you want to enable interactive services on. If `nodes` are not selected, by default, interactive applications are only enabled on the head node. Values are "all", or compute node index (for ex. "0", "1" etc.)
  my_jupyter_lab:
    type: jupyter_lab
    nodes: all
display_name: pull_dataset_from_huggingface
experiment_name: pull_dataset
description: Pull dataset from huggingface to Azure Blob

Overwriting pull_dataset_from_huggingface.yaml


### Submit AML job using the generated YAML file

In [15]:
!az ml job create -f pull_dataset_from_huggingface.yaml

{
  "code": "azureml:/subscriptions/781b03e7-6eb7-4506-bab8-cf3a0d89b1d4/resourceGroups/antonslutsky-rg/providers/Microsoft.MachineLearningServices/workspaces/gpu-workspace-uksouth/codes/d48fc6f4-2686-455c-af96-024483f3891c/versions/1",
  "command": "./pull_dataset_from_huggingface.sh ${{inputs.huggingface_dataset_name}} ${{outputs.output_dir_path}}",
  "compute": "azureml:cpu-cluster",
  "creation_context": {
    "created_at": "2024-10-28T17:56:16.968168+00:00",
    "created_by": "Anton Slutsky",
    "created_by_type": "User"
  },
  "description": "Pull dataset from huggingface to Azure Blob",
  "display_name": "pull_dataset_from_huggingface",
  "environment": "azureml:CliV2AnonymousEnvironment:da9c6c8442d99d9a1652bf9ddcc136a0",
  "environment_variables": {},
  "experiment_name": "pull_dataset",
  "id": "azureml:/subscriptions/781b03e7-6eb7-4506-bab8-cf3a0d89b1d4/resourceGroups/antonslutsky-rg/providers/Microsoft.MachineLearningServices/workspaces/gpu-workspace-uksouth/jobs/patient_po

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.

Uploading src (0.0 MBs): 100%|#######